In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS main")
spark.sql("CREATE SCHEMA IF NOT EXISTS main.retail")
spark.sql("CREATE VOLUME IF NOT EXISTS main.retail.data")

DELTA_PATH   = "/Volumes/main/retail/data/delta"
PARQUET_PATH = "/Volumes/main/retail/data/parquet"

In [0]:
products_data = [
(101,"Rice Bag","Groceries","Hyderabad",1200,50),
(102,"Wheat Flour","Groceries","Bengaluru",900,80),
(103,"Sunflower Oil","Groceries","Mumbai",1800,40),
(104,"Milk Pack","Dairy","Chennai",60,200),
(105,"Cheese Block","Dairy","Delhi",450,70),
(106,"Soap","Personal Care","Kolkata",120,300),
(107,"Shampoo","Personal Care","Pune",320,150),
(108,"Toothpaste","Personal Care","Ahmedabad",90,250),
(109,"Notebook","Stationery","Hyderabad",75,500),
(110,"Pen Pack","Stationery","Mumbai",110,400),
(111,"LED TV","Electronics","Delhi",45000,15),
(112,"Refrigerator","Electronics","Chennai",38000,10),
(113,"Washing Machine","Electronics","Bengaluru",29000,12),
(114,"Mobile Phone","Electronics","Hyderabad",25000,35),
(115,"Laptop","Electronics","Pune",62000,18),
(116,"Air Conditioner","Electronics","Mumbai",42000,9),
(117,"Mixer Grinder","Home Appliances","Kolkata",3500,45),
(118,"Water Purifier","Home Appliances","Delhi",12000,20),
(119,"Ceiling Fan","Home Appliances","Ahmedabad",2800,60),
(120,"Gas Stove","Home Appliances","Chennai",5500,25)
]
products_columns = ["product_id","product_name","category","warehouse_city","price","stock_quantity"]
products_df = spark.createDataFrame(products_data, products_columns)

suppliers_data = [
(201,"Reddy Traders","Hyderabad","Groceries"),
(202,"Fresh Dairy Ltd","Chennai","Dairy"),
(203,"CarePlus Suppliers","Mumbai","Personal Care"),
(204,"Elite Electronics","Delhi","Electronics"),
(205,"OfficeKart","Bengaluru","Stationery"),
(206,"HomeNeeds Pvt Ltd","Pune","Home Appliances"),
(207,"National Grocers","Ahmedabad","Groceries"),
(208,"Smart Electronics","Kolkata","Electronics"),
(209,"Daily Essentials","Hyderabad","Personal Care"),
(210,"Kitchen World","Chennai","Home Appliances")
]
suppliers_columns = ["supplier_id","supplier_name","supplier_city","specialization"]
suppliers_df = spark.createDataFrame(suppliers_data, suppliers_columns)

orders_data = [
(301,101,201,"2024-04-01",20,"Delivered"),
(302,102,201,"2024-04-01",35,"Delivered"),
(303,111,204,"2024-04-02",2,"Delivered"),
(304,114,208,"2024-04-02",5,"Pending"),
(305,115,204,"2024-04-03",3,"Delivered"),
(306,104,202,"2024-04-03",50,"Delivered"),
(307,105,202,"2024-04-04",18,"Cancelled"),
(308,117,206,"2024-04-04",7,"Delivered"),
(309,118,206,"2024-04-05",4,"Pending"),
(310,119,206,"2024-04-05",12,"Delivered"),
(311,120,210,"2024-04-06",6,"Delivered"),
(312,113,204,"2024-04-06",4,"Delivered"),
(313,116,208,"2024-04-07",2,"Pending"),
(314,109,205,"2024-04-07",80,"Delivered"),
(315,110,205,"2024-04-08",120,"Delivered"),
(316,106,203,"2024-04-08",60,"Cancelled"),
(317,107,209,"2024-04-09",25,"Delivered"),
(318,108,203,"2024-04-09",40,"Delivered"),
(319,112,208,"2024-04-10",2,"Pending"),
(320,101,207,"2024-04-10",15,"Delivered")
]
orders_columns = ["order_id","product_id","supplier_id","order_date","quantity","order_status"]
orders_df = spark.createDataFrame(orders_data, orders_columns)

payments_data = [
(401,301,24000,"UPI","Paid"),
(402,302,31500,"Credit Card","Paid"),
(403,303,90000,"Bank Transfer","Paid"),
(404,304,125000,"UPI","Pending"),
(405,305,186000,"Bank Transfer","Paid"),
(406,306,3000,"Cash","Paid"),
(407,307,8100,"UPI","Cancelled"),
(408,308,24500,"Debit Card","Paid"),
(409,309,48000,"UPI","Pending"),
(410,310,33600,"Cash","Paid"),
(411,311,33000,"Credit Card","Paid"),
(412,312,116000,"Bank Transfer","Paid"),
(413,313,84000,"UPI","Pending"),
(414,314,6000,"Cash","Paid"),
(415,315,13200,"UPI","Paid"),
(416,316,7200,"Cash","Cancelled"),
(417,317,8000,"UPI","Paid"),
(418,318,3600,"Debit Card","Paid"),
(419,319,76000,"Bank Transfer","Pending"),
(420,320,18000,"UPI","Paid")
]
payments_columns = ["payment_id","order_id","bill_amount","payment_mode","payment_status"]
payments_df = spark.createDataFrame(payments_data, payments_columns)

Part 1 — DataFrame Fundamentals

In [0]:
# 1
products_df.show()
suppliers_df.show()
orders_df.show()
payments_df.show()

+----------+---------------+---------------+--------------+-----+--------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|
+----------+---------------+---------------+--------------+-----+--------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|
|       106|           Soap|  Personal Care|       Kolkata|  120|           300|
|       107|        Shampoo|  Personal Care|          Pune|  320|           150|
|       108|     Toothpaste|  Personal Care|     Ahmedabad|   90|           250|
|       109|       Notebook|     Stationery|     Hyderabad|   75|           500|
|       110|       Pen Pack|

In [0]:
# 2
products_df.printSchema()
suppliers_df.printSchema()
orders_df.printSchema()
payments_df.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- warehouse_city: string (nullable = true)
 |-- price: long (nullable = true)
 |-- stock_quantity: long (nullable = true)

root
 |-- supplier_id: long (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- supplier_city: string (nullable = true)
 |-- specialization: string (nullable = true)

root
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- supplier_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- order_status: string (nullable = true)

root
 |-- payment_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- bill_amount: long (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)



In [0]:
# 3
print(products_df.count())
print(suppliers_df.count())
print(orders_df.count())
print(payments_df.count())

20
10
20
20


In [0]:
# 4
products_df.show(10)

+----------+-------------+-------------+--------------+-----+--------------+
|product_id| product_name|     category|warehouse_city|price|stock_quantity|
+----------+-------------+-------------+--------------+-----+--------------+
|       101|     Rice Bag|    Groceries|     Hyderabad| 1200|            50|
|       102|  Wheat Flour|    Groceries|     Bengaluru|  900|            80|
|       103|Sunflower Oil|    Groceries|        Mumbai| 1800|            40|
|       104|    Milk Pack|        Dairy|       Chennai|   60|           200|
|       105| Cheese Block|        Dairy|         Delhi|  450|            70|
|       106|         Soap|Personal Care|       Kolkata|  120|           300|
|       107|      Shampoo|Personal Care|          Pune|  320|           150|
|       108|   Toothpaste|Personal Care|     Ahmedabad|   90|           250|
|       109|     Notebook|   Stationery|     Hyderabad|   75|           500|
|       110|     Pen Pack|   Stationery|        Mumbai|  110|           400|

In [0]:
# 5
products_df.select("product_name","category","stock_quantity").show()

+---------------+---------------+--------------+
|   product_name|       category|stock_quantity|
+---------------+---------------+--------------+
|       Rice Bag|      Groceries|            50|
|    Wheat Flour|      Groceries|            80|
|  Sunflower Oil|      Groceries|            40|
|      Milk Pack|          Dairy|           200|
|   Cheese Block|          Dairy|            70|
|           Soap|  Personal Care|           300|
|        Shampoo|  Personal Care|           150|
|     Toothpaste|  Personal Care|           250|
|       Notebook|     Stationery|           500|
|       Pen Pack|     Stationery|           400|
|         LED TV|    Electronics|            15|
|   Refrigerator|    Electronics|            10|
|Washing Machine|    Electronics|            12|
|   Mobile Phone|    Electronics|            35|
|         Laptop|    Electronics|            18|
|Air Conditioner|    Electronics|             9|
|  Mixer Grinder|Home Appliances|            45|
| Water Purifier|Hom

In [0]:
# 6
suppliers_df.filter(suppliers_df.supplier_city.isin("Hyderabad","Chennai")).show()

+-----------+----------------+-------------+---------------+
|supplier_id|   supplier_name|supplier_city| specialization|
+-----------+----------------+-------------+---------------+
|        201|   Reddy Traders|    Hyderabad|      Groceries|
|        202| Fresh Dairy Ltd|      Chennai|          Dairy|
|        209|Daily Essentials|    Hyderabad|  Personal Care|
|        210|   Kitchen World|      Chennai|Home Appliances|
+-----------+----------------+-------------+---------------+



In [0]:
# 7
orders_df.filter(orders_df.order_status == "Delivered").show()

+--------+----------+-----------+----------+--------+------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|
+--------+----------+-----------+----------+--------+------------+
|     301|       101|        201|2024-04-01|      20|   Delivered|
|     302|       102|        201|2024-04-01|      35|   Delivered|
|     303|       111|        204|2024-04-02|       2|   Delivered|
|     305|       115|        204|2024-04-03|       3|   Delivered|
|     306|       104|        202|2024-04-03|      50|   Delivered|
|     308|       117|        206|2024-04-04|       7|   Delivered|
|     310|       119|        206|2024-04-05|      12|   Delivered|
|     311|       120|        210|2024-04-06|       6|   Delivered|
|     312|       113|        204|2024-04-06|       4|   Delivered|
|     314|       109|        205|2024-04-07|      80|   Delivered|
|     315|       110|        205|2024-04-08|     120|   Delivered|
|     317|       107|        209|2024-04-09|      25|   Delive

In [0]:
# 8
orders_df.filter(orders_df.order_status == "Pending").show()

+--------+----------+-----------+----------+--------+------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|
+--------+----------+-----------+----------+--------+------------+
|     304|       114|        208|2024-04-02|       5|     Pending|
|     309|       118|        206|2024-04-05|       4|     Pending|
|     313|       116|        208|2024-04-07|       2|     Pending|
|     319|       112|        208|2024-04-10|       2|     Pending|
+--------+----------+-----------+----------+--------+------------+



In [0]:
# 9
products_df.filter(products_df.category == "Electronics").show()

+----------+---------------+-----------+--------------+-----+--------------+
|product_id|   product_name|   category|warehouse_city|price|stock_quantity|
+----------+---------------+-----------+--------------+-----+--------------+
|       111|         LED TV|Electronics|         Delhi|45000|            15|
|       112|   Refrigerator|Electronics|       Chennai|38000|            10|
|       113|Washing Machine|Electronics|     Bengaluru|29000|            12|
|       114|   Mobile Phone|Electronics|     Hyderabad|25000|            35|
|       115|         Laptop|Electronics|          Pune|62000|            18|
|       116|Air Conditioner|Electronics|        Mumbai|42000|             9|
+----------+---------------+-----------+--------------+-----+--------------+



In [0]:
# 10
products_df.filter(products_df.stock_quantity < 20).show()

+----------+---------------+-----------+--------------+-----+--------------+
|product_id|   product_name|   category|warehouse_city|price|stock_quantity|
+----------+---------------+-----------+--------------+-----+--------------+
|       111|         LED TV|Electronics|         Delhi|45000|            15|
|       112|   Refrigerator|Electronics|       Chennai|38000|            10|
|       113|Washing Machine|Electronics|     Bengaluru|29000|            12|
|       115|         Laptop|Electronics|          Pune|62000|            18|
|       116|Air Conditioner|Electronics|        Mumbai|42000|             9|
+----------+---------------+-----------+--------------+-----+--------------+



Part 2 — DataFrame Transformations

In [0]:
# 11
from pyspark.sql.functions import to_date
orders_df = orders_df.withColumn("order_date", to_date("order_date","yyyy-MM-dd"))
orders_df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- supplier_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: long (nullable = true)
 |-- order_status: string (nullable = true)



In [0]:
# 12
from pyspark.sql.functions import col
orders_df = orders_df.withColumn("total_order_value", col("quantity") * col("quantity"))
from pyspark.sql.functions import col as c
products_orders = orders_df.join(products_df, on="product_id", how="left")
orders_df = orders_df.join(products_df.select("product_id","price"), on="product_id", how="left") \
    .withColumn("total_order_value", col("quantity") * col("price")) \
    .drop("price")
orders_df.show()

+----------+--------+-----------+----------+--------+------------+-----------------+
|product_id|order_id|supplier_id|order_date|quantity|order_status|total_order_value|
+----------+--------+-----------+----------+--------+------------+-----------------+
|       101|     301|        201|2024-04-01|      20|   Delivered|            24000|
|       102|     302|        201|2024-04-01|      35|   Delivered|            31500|
|       111|     303|        204|2024-04-02|       2|   Delivered|            90000|
|       114|     304|        208|2024-04-02|       5|     Pending|           125000|
|       115|     305|        204|2024-04-03|       3|   Delivered|           186000|
|       104|     306|        202|2024-04-03|      50|   Delivered|             3000|
|       105|     307|        202|2024-04-04|      18|   Cancelled|             8100|
|       117|     308|        206|2024-04-04|       7|   Delivered|            24500|
|       118|     309|        206|2024-04-05|       4|     Pending

In [0]:
# 13
from pyspark.sql.functions import when
products_df = products_df.withColumn("stock_status",
    when(col("stock_quantity") < 20, "Low")
    .when(col("stock_quantity") < 50, "Medium")
    .otherwise("High"))
products_df.show()

+----------+---------------+---------------+--------------+-----+--------------+------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|stock_status|
+----------+---------------+---------------+--------------+-----+--------------+------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|        High|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|        High|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|      Medium|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|        High|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|        High|
|       106|           Soap|  Personal Care|       Kolkata|  120|           300|        High|
|       107|        Shampoo|  Personal Care|          Pune|  320|           150|        High|
|       108|     Toothpaste|  Personal Care|     Ahmedabad| 

In [0]:
# 14
orders_df = orders_df.withColumn("order_priority",
    when(col("order_status") == "Pending", "High")
    .when(col("order_status") == "Delivered", "Low")
    .otherwise("Cancelled"))
orders_df.show()

+----------+--------+-----------+----------+--------+------------+-----------------+--------------+
|product_id|order_id|supplier_id|order_date|quantity|order_status|total_order_value|order_priority|
+----------+--------+-----------+----------+--------+------------+-----------------+--------------+
|       101|     301|        201|2024-04-01|      20|   Delivered|            24000|           Low|
|       102|     302|        201|2024-04-01|      35|   Delivered|            31500|           Low|
|       111|     303|        204|2024-04-02|       2|   Delivered|            90000|           Low|
|       114|     304|        208|2024-04-02|       5|     Pending|           125000|          High|
|       115|     305|        204|2024-04-03|       3|   Delivered|           186000|           Low|
|       104|     306|        202|2024-04-03|      50|   Delivered|             3000|           Low|
|       105|     307|        202|2024-04-04|      18|   Cancelled|             8100|     Cancelled|


In [0]:
# 15
products_df = products_df.withColumn("expensive_product_flag",
    when(col("price") > 10000, "Yes").otherwise("No"))
products_df.show()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|stock_status|expensive_product_flag|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|        High|                    No|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|        High|                    No|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|      Medium|                    No|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|        High|                    No|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|        High|                    No|
|       106|           Soap|  Personal Care|       Kolkata|  120

In [0]:
# 16
from pyspark.sql.functions import upper
products_df = products_df.withColumn("category", upper(col("category")))
products_df.show()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|stock_status|expensive_product_flag|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|        High|                    No|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|        High|                    No|
|       103|  Sunflower Oil|      GROCERIES|        Mumbai| 1800|            40|      Medium|                    No|
|       104|      Milk Pack|          DAIRY|       Chennai|   60|           200|        High|                    No|
|       105|   Cheese Block|          DAIRY|         Delhi|  450|            70|        High|                    No|
|       106|           Soap|  PERSONAL CARE|       Kolkata|  120

In [0]:
# 17
products_df = products_df.withColumnRenamed("warehouse_city","inventory_city")
products_df.show()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+
|product_id|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|        High|                    No|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|        High|                    No|
|       103|  Sunflower Oil|      GROCERIES|        Mumbai| 1800|            40|      Medium|                    No|
|       104|      Milk Pack|          DAIRY|       Chennai|   60|           200|        High|                    No|
|       105|   Cheese Block|          DAIRY|         Delhi|  450|            70|        High|                    No|
|       106|           Soap|  PERSONAL CARE|       Kolkata|  120

In [0]:
# 18
products_df = products_df.withColumn("inventory_value", col("price") * col("stock_quantity"))
products_df.show()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+
|product_id|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|        High|                    No|          60000|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|        High|                    No|          72000|
|       103|  Sunflower Oil|      GROCERIES|        Mumbai| 1800|            40|      Medium|                    No|          72000|
|       104|      Milk Pack|          DAIRY|       Chennai|   60|           200|        High|                    No|          12000|
|       105|   Cheese Block|          DAIRY|         Delhi|  450|    

In [0]:
# 19
orders_df = orders_df.drop("total_order_value")
orders_df.show()

+----------+--------+-----------+----------+--------+------------+--------------+
|product_id|order_id|supplier_id|order_date|quantity|order_status|order_priority|
+----------+--------+-----------+----------+--------+------------+--------------+
|       101|     301|        201|2024-04-01|      20|   Delivered|           Low|
|       102|     302|        201|2024-04-01|      35|   Delivered|           Low|
|       111|     303|        204|2024-04-02|       2|   Delivered|           Low|
|       114|     304|        208|2024-04-02|       5|     Pending|          High|
|       115|     305|        204|2024-04-03|       3|   Delivered|           Low|
|       104|     306|        202|2024-04-03|      50|   Delivered|           Low|
|       105|     307|        202|2024-04-04|      18|   Cancelled|     Cancelled|
|       117|     308|        206|2024-04-04|       7|   Delivered|           Low|
|       118|     309|        206|2024-04-05|       4|     Pending|          High|
|       119|    

In [0]:
# 20
products_df = products_df.withColumn("low_stock_flag",when(col("stock_quantity") < 20, "Yes").otherwise("No"))
products_df.show()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+
|product_id|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|        High|                    No|          60000|            No|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|        High|                    No|          72000|            No|
|       103|  Sunflower Oil|      GROCERIES|        Mumbai| 1800|            40|      Medium|                    No|          72000|            No|
|       104|      Milk Pack|          DAIRY|       Chennai|   60|           200|        High|                   

Part 3 — Joins

In [0]:
# 21
products_orders_df = products_df.join(orders_df, on="product_id", how="inner")
products_orders_df.show()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+--------+-----------+----------+--------+------------+--------------+
|product_id|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|order_id|supplier_id|order_date|quantity|order_status|order_priority|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+--------+-----------+----------+--------+------------+--------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|        High|                    No|          60000|            No|     301|        201|2024-04-01|      20|   Delivered|           Low|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|        High|                    No|          72000|

In [0]:
# 22
orders_suppliers_df = orders_df.join(suppliers_df, on="supplier_id", how="inner")
orders_suppliers_df.show()

+-----------+----------+--------+----------+--------+------------+--------------+------------------+-------------+---------------+
|supplier_id|product_id|order_id|order_date|quantity|order_status|order_priority|     supplier_name|supplier_city| specialization|
+-----------+----------+--------+----------+--------+------------+--------------+------------------+-------------+---------------+
|        201|       101|     301|2024-04-01|      20|   Delivered|           Low|     Reddy Traders|    Hyderabad|      Groceries|
|        201|       102|     302|2024-04-01|      35|   Delivered|           Low|     Reddy Traders|    Hyderabad|      Groceries|
|        204|       111|     303|2024-04-02|       2|   Delivered|           Low| Elite Electronics|        Delhi|    Electronics|
|        208|       114|     304|2024-04-02|       5|     Pending|          High| Smart Electronics|      Kolkata|    Electronics|
|        204|       115|     305|2024-04-03|       3|   Delivered|           Low| E

In [0]:
# 23
orders_payments_df = orders_df.join(payments_df, on="order_id", how="inner")
orders_payments_df.show()

+--------+----------+-----------+----------+--------+------------+--------------+----------+-----------+-------------+--------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|order_priority|payment_id|bill_amount| payment_mode|payment_status|
+--------+----------+-----------+----------+--------+------------+--------------+----------+-----------+-------------+--------------+
|     301|       101|        201|2024-04-01|      20|   Delivered|           Low|       401|      24000|          UPI|          Paid|
|     302|       102|        201|2024-04-01|      35|   Delivered|           Low|       402|      31500|  Credit Card|          Paid|
|     303|       111|        204|2024-04-02|       2|   Delivered|           Low|       403|      90000|Bank Transfer|          Paid|
|     304|       114|        208|2024-04-02|       5|     Pending|          High|       404|     125000|          UPI|       Pending|
|     305|       115|        204|2024-04-03|       3|   Delive

In [0]:
# 24
final_df = orders_df .join(products_df, on="product_id", how="inner").join(suppliers_df, on="supplier_id", how="inner").join(payments_df, on="order_id", how="inner")
final_df.show()

+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|     301|        201|       101|2024-04-01|  

In [0]:
# 25
final_df.select("product_name","supplier_name","quantity","bill_amount").show()

+---------------+------------------+--------+-----------+
|   product_name|     supplier_name|quantity|bill_amount|
+---------------+------------------+--------+-----------+
|       Rice Bag|     Reddy Traders|      20|      24000|
|    Wheat Flour|     Reddy Traders|      35|      31500|
|         LED TV| Elite Electronics|       2|      90000|
|   Mobile Phone| Smart Electronics|       5|     125000|
|         Laptop| Elite Electronics|       3|     186000|
|      Milk Pack|   Fresh Dairy Ltd|      50|       3000|
|   Cheese Block|   Fresh Dairy Ltd|      18|       8100|
|  Mixer Grinder| HomeNeeds Pvt Ltd|       7|      24500|
| Water Purifier| HomeNeeds Pvt Ltd|       4|      48000|
|    Ceiling Fan| HomeNeeds Pvt Ltd|      12|      33600|
|      Gas Stove|     Kitchen World|       6|      33000|
|Washing Machine| Elite Electronics|       4|     116000|
|Air Conditioner| Smart Electronics|       2|      84000|
|       Notebook|        OfficeKart|      80|       6000|
|       Pen Pa

In [0]:
# 26
final_df.filter(col("inventory_city") != col("supplier_city")) \
    .select("supplier_name","supplier_city","inventory_city").show()

+------------------+-------------+--------------+
|     supplier_name|supplier_city|inventory_city|
+------------------+-------------+--------------+
|     Reddy Traders|    Hyderabad|     Bengaluru|
| Smart Electronics|      Kolkata|     Hyderabad|
| Elite Electronics|        Delhi|          Pune|
|   Fresh Dairy Ltd|      Chennai|         Delhi|
| HomeNeeds Pvt Ltd|         Pune|       Kolkata|
| HomeNeeds Pvt Ltd|         Pune|         Delhi|
| HomeNeeds Pvt Ltd|         Pune|     Ahmedabad|
| Elite Electronics|        Delhi|     Bengaluru|
| Smart Electronics|      Kolkata|        Mumbai|
|        OfficeKart|    Bengaluru|     Hyderabad|
|        OfficeKart|    Bengaluru|        Mumbai|
|CarePlus Suppliers|       Mumbai|       Kolkata|
|  Daily Essentials|    Hyderabad|          Pune|
|CarePlus Suppliers|       Mumbai|     Ahmedabad|
| Smart Electronics|      Kolkata|       Chennai|
|  National Grocers|    Ahmedabad|     Hyderabad|
+------------------+-------------+--------------+


In [0]:
# 27
final_df.filter((col("order_status") == "Delivered") & (col("payment_status") == "Paid")).show()

+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|     301|        201|       101|2024-04-01|  

In [0]:
# 28
final_df.filter((col("order_status") == "Pending") & (col("payment_status") == "Pending")).show()

+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+-----------------+-------------+---------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|    supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+-----------------+-------------+---------------+----------+-----------+-------------+--------------+
|     304|        208|       114|2024-04-02|     

In [0]:
# 29
final_df.filter((col("order_status") == "Cancelled") & (col("payment_status") == "Cancelled")).show()

+--------+-----------+----------+----------+--------+------------+--------------+------------+-------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+------------------+-------------+--------------+----------+-----------+------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|product_name|     category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|     supplier_name|supplier_city|specialization|payment_id|bill_amount|payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+--------------+------------+-------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+------------------+-------------+--------------+----------+-----------+------------+--------------+
|     307|        202|       105|2024-04-04|      18|   Cancelled| 

In [0]:
# 30
from pyspark.sql.functions import count
final_df.groupBy("product_id","product_name").agg(count("order_id").alias("order_count")).filter(col("order_count") > 1).show()

+----------+------------+-----------+
|product_id|product_name|order_count|
+----------+------------+-----------+
|       101|    Rice Bag|          2|
+----------+------------+-----------+



Part 4 — Aggregations

In [0]:
# 31
products_df.groupBy("category").count().show()

+---------------+-----+
|       category|count|
+---------------+-----+
|      GROCERIES|    3|
|          DAIRY|    2|
|  PERSONAL CARE|    3|
|     STATIONERY|    2|
|    ELECTRONICS|    6|
|HOME APPLIANCES|    4|
+---------------+-----+



In [0]:
# 32
orders_df.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|   Delivered|   14|
|     Pending|    4|
|   Cancelled|    2|
+------------+-----+



In [0]:
# 33
suppliers_df.groupBy("supplier_city").count().show()

+-------------+-----+
|supplier_city|count|
+-------------+-----+
|    Hyderabad|    2|
|      Chennai|    2|
|       Mumbai|    1|
|        Delhi|    1|
|    Bengaluru|    1|
|         Pune|    1|
|    Ahmedabad|    1|
|      Kolkata|    1|
+-------------+-----+



In [0]:
# 34
from pyspark.sql.functions import sum
payments_df.agg(sum("bill_amount").alias("total_revenue")).show()

+-------------+
|total_revenue|
+-------------+
|       938700|
+-------------+



In [0]:
# 35
from pyspark.sql.functions import avg
payments_df.agg(avg("bill_amount").alias("avg_bill")).show()

+--------+
|avg_bill|
+--------+
| 46935.0|
+--------+



In [0]:
# 36
final_df.groupBy("category").agg(sum("bill_amount").alias("total_revenue")).show()

+---------------+-------------+
|       category|total_revenue|
+---------------+-------------+
|      GROCERIES|        73500|
|    ELECTRONICS|       677000|
|          DAIRY|        11100|
|HOME APPLIANCES|       139100|
|     STATIONERY|        19200|
|  PERSONAL CARE|        18800|
+---------------+-------------+



In [0]:
# 37
final_df.groupBy("supplier_name").agg(sum("bill_amount").alias("total_revenue")).show()

+------------------+-------------+
|     supplier_name|total_revenue|
+------------------+-------------+
|     Reddy Traders|        55500|
| Smart Electronics|       285000|
| Elite Electronics|       392000|
|   Fresh Dairy Ltd|        11100|
| HomeNeeds Pvt Ltd|       106100|
|     Kitchen World|        33000|
|        OfficeKart|        19200|
|CarePlus Suppliers|        10800|
|  Daily Essentials|         8000|
|  National Grocers|        18000|
+------------------+-------------+



In [0]:
# 38
final_df.groupBy("inventory_city").agg(sum("bill_amount").alias("total_revenue")).show()

+--------------+-------------+
|inventory_city|total_revenue|
+--------------+-------------+
|     Hyderabad|       173000|
|     Bengaluru|       147500|
|         Delhi|       146100|
|          Pune|       194000|
|       Chennai|       112000|
|     Ahmedabad|        37200|
|       Kolkata|        31700|
|        Mumbai|        97200|
+--------------+-------------+



In [0]:
# 39
final_df.groupBy("product_id","product_name").agg(sum("quantity").alias("total_qty")).orderBy(col("total_qty").desc()).show()

+----------+---------------+---------+
|product_id|   product_name|total_qty|
+----------+---------------+---------+
|       110|       Pen Pack|      120|
|       109|       Notebook|       80|
|       106|           Soap|       60|
|       104|      Milk Pack|       50|
|       108|     Toothpaste|       40|
|       101|       Rice Bag|       35|
|       102|    Wheat Flour|       35|
|       107|        Shampoo|       25|
|       105|   Cheese Block|       18|
|       119|    Ceiling Fan|       12|
|       117|  Mixer Grinder|        7|
|       120|      Gas Stove|        6|
|       114|   Mobile Phone|        5|
|       118| Water Purifier|        4|
|       113|Washing Machine|        4|
|       115|         Laptop|        3|
|       111|         LED TV|        2|
|       116|Air Conditioner|        2|
|       112|   Refrigerator|        2|
+----------+---------------+---------+



In [0]:
# 40
final_df.groupBy("product_id","product_name").agg(sum("quantity").alias("total_qty")).orderBy(col("total_qty").asc()).show()

+----------+---------------+---------+
|product_id|   product_name|total_qty|
+----------+---------------+---------+
|       111|         LED TV|        2|
|       116|Air Conditioner|        2|
|       112|   Refrigerator|        2|
|       115|         Laptop|        3|
|       118| Water Purifier|        4|
|       113|Washing Machine|        4|
|       114|   Mobile Phone|        5|
|       120|      Gas Stove|        6|
|       117|  Mixer Grinder|        7|
|       119|    Ceiling Fan|       12|
|       105|   Cheese Block|       18|
|       107|        Shampoo|       25|
|       101|       Rice Bag|       35|
|       102|    Wheat Flour|       35|
|       108|     Toothpaste|       40|
|       104|      Milk Pack|       50|
|       106|           Soap|       60|
|       109|       Notebook|       80|
|       110|       Pen Pack|      120|
+----------+---------------+---------+



Part 5 — Spark SQL

In [0]:
# 41
products_df.createOrReplaceTempView("products")
suppliers_df.createOrReplaceTempView("suppliers")
orders_df.createOrReplaceTempView("orders")
payments_df.createOrReplaceTempView("payments")
final_df.createOrReplaceTempView("final")

In [0]:
# 42
spark.sql("SELECT * FROM products").show()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+
|product_id|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|        High|                    No|          60000|            No|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|        High|                    No|          72000|            No|
|       103|  Sunflower Oil|      GROCERIES|        Mumbai| 1800|            40|      Medium|                    No|          72000|            No|
|       104|      Milk Pack|          DAIRY|       Chennai|   60|           200|        High|                   

In [0]:
# 43
spark.sql("""
SELECT p.product_name, o.order_date, o.order_status, o.quantity
FROM orders o
JOIN products p ON o.product_id = p.product_id
WHERE p.category = 'ELECTRONICS'
""").show()

+---------------+----------+------------+--------+
|   product_name|order_date|order_status|quantity|
+---------------+----------+------------+--------+
|         LED TV|2024-04-02|   Delivered|       2|
|   Mobile Phone|2024-04-02|     Pending|       5|
|         Laptop|2024-04-03|   Delivered|       3|
|Washing Machine|2024-04-06|   Delivered|       4|
|Air Conditioner|2024-04-07|     Pending|       2|
|   Refrigerator|2024-04-10|     Pending|       2|
+---------------+----------+------------+--------+



In [0]:
# 44
spark.sql("SELECT category, SUM(bill_amount) AS revenue FROM final GROUP BY category").show()

+---------------+-------+
|       category|revenue|
+---------------+-------+
|      GROCERIES|  73500|
|    ELECTRONICS| 677000|
|          DAIRY|  11100|
|HOME APPLIANCES| 139100|
|     STATIONERY|  19200|
|  PERSONAL CARE|  18800|
+---------------+-------+



In [0]:
# 45
spark.sql("SELECT supplier_name, SUM(bill_amount) AS revenue FROM final GROUP BY supplier_name").show()

+------------------+-------+
|     supplier_name|revenue|
+------------------+-------+
|     Reddy Traders|  55500|
| Smart Electronics| 285000|
| Elite Electronics| 392000|
|   Fresh Dairy Ltd|  11100|
| HomeNeeds Pvt Ltd| 106100|
|     Kitchen World|  33000|
|        OfficeKart|  19200|
|CarePlus Suppliers|  10800|
|  Daily Essentials|   8000|
|  National Grocers|  18000|
+------------------+-------+



In [0]:
# 46
spark.sql("SELECT * FROM final ORDER BY bill_amount DESC LIMIT 5").show()

+--------+-----------+----------+----------+--------+------------+--------------+---------------+-----------+--------------+-----+--------------+------------+----------------------+---------------+--------------+-----------------+-------------+--------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|   category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|    supplier_name|supplier_city|specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+-----------+--------------+-----+--------------+------------+----------------------+---------------+--------------+-----------------+-------------+--------------+----------+-----------+-------------+--------------+
|     305|        204|       115|2024-04-03|       3|   Delivere

In [0]:
# 47
spark.sql("SELECT supplier_name, COUNT(order_id) AS order_count FROM final GROUP BY supplier_name").show()

+------------------+-----------+
|     supplier_name|order_count|
+------------------+-----------+
|     Reddy Traders|          2|
| Smart Electronics|          3|
| Elite Electronics|          3|
|   Fresh Dairy Ltd|          2|
| HomeNeeds Pvt Ltd|          3|
|     Kitchen World|          1|
|        OfficeKart|          2|
|CarePlus Suppliers|          2|
|  Daily Essentials|          1|
|  National Grocers|          1|
+------------------+-----------+



In [0]:
# 48
spark.sql("SELECT category, COUNT(order_id) AS order_count FROM final GROUP BY category").show()

+---------------+-----------+
|       category|order_count|
+---------------+-----------+
|      GROCERIES|          3|
|    ELECTRONICS|          6|
|          DAIRY|          2|
|HOME APPLIANCES|          4|
|     STATIONERY|          2|
|  PERSONAL CARE|          3|
+---------------+-----------+



In [0]:
# 49
spark.sql("SELECT payment_mode, AVG(bill_amount) AS avg_payment FROM final GROUP BY payment_mode").show()

+-------------+-----------+
| payment_mode|avg_payment|
+-------------+-----------+
|  Credit Card|    32250.0|
|          UPI|    41037.5|
|Bank Transfer|   117000.0|
|         Cash|    12450.0|
|   Debit Card|    14050.0|
+-------------+-----------+



In [0]:
# 50
spark.sql("SELECT product_name, SUM(bill_amount) AS total_revenue FROM final GROUP BY product_name HAVING SUM(bill_amount) > 100000").show()

+---------------+-------------+
|   product_name|total_revenue|
+---------------+-------------+
|         Laptop|       186000|
|   Mobile Phone|       125000|
|Washing Machine|       116000|
+---------------+-------------+



Part 6 — Window Functions

In [0]:
# 51
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

cat_window = Window.partitionBy("category").orderBy(col("total_revenue").desc())
prod_rev_df = final_df.groupBy("product_id","product_name","category").agg(sum("bill_amount").alias("total_revenue"))
prod_rev_df.withColumn("rank", rank().over(cat_window)).show()

+----------+---------------+---------------+-------------+----+
|product_id|   product_name|       category|total_revenue|rank|
+----------+---------------+---------------+-------------+----+
|       105|   Cheese Block|          DAIRY|         8100|   1|
|       104|      Milk Pack|          DAIRY|         3000|   2|
|       115|         Laptop|    ELECTRONICS|       186000|   1|
|       114|   Mobile Phone|    ELECTRONICS|       125000|   2|
|       113|Washing Machine|    ELECTRONICS|       116000|   3|
|       111|         LED TV|    ELECTRONICS|        90000|   4|
|       116|Air Conditioner|    ELECTRONICS|        84000|   5|
|       112|   Refrigerator|    ELECTRONICS|        76000|   6|
|       101|       Rice Bag|      GROCERIES|        42000|   1|
|       102|    Wheat Flour|      GROCERIES|        31500|   2|
|       118| Water Purifier|HOME APPLIANCES|        48000|   1|
|       119|    Ceiling Fan|HOME APPLIANCES|        33600|   2|
|       120|      Gas Stove|HOME APPLIAN

In [0]:
# 52
city_window = Window.partitionBy("supplier_city").orderBy(col("total_revenue").desc())
supp_rev_df = final_df.groupBy("supplier_id","supplier_name","supplier_city").agg(sum("bill_amount").alias("total_revenue"))
supp_rev_df.withColumn("rank", rank().over(city_window)).show()

+-----------+------------------+-------------+-------------+----+
|supplier_id|     supplier_name|supplier_city|total_revenue|rank|
+-----------+------------------+-------------+-------------+----+
|        207|  National Grocers|    Ahmedabad|        18000|   1|
|        205|        OfficeKart|    Bengaluru|        19200|   1|
|        210|     Kitchen World|      Chennai|        33000|   1|
|        202|   Fresh Dairy Ltd|      Chennai|        11100|   2|
|        204| Elite Electronics|        Delhi|       392000|   1|
|        201|     Reddy Traders|    Hyderabad|        55500|   1|
|        209|  Daily Essentials|    Hyderabad|         8000|   2|
|        208| Smart Electronics|      Kolkata|       285000|   1|
|        203|CarePlus Suppliers|       Mumbai|        10800|   1|
|        206| HomeNeeds Pvt Ltd|         Pune|       106100|   1|
+-----------+------------------+-------------+-------------+----+



In [0]:
# 53
from pyspark.sql.functions import row_number
row_window = Window.partitionBy("category").orderBy(col("total_revenue").desc())
prod_rev_df.withColumn("row_num", row_number().over(row_window)).filter(col("row_num") == 1).show()

+----------+--------------+---------------+-------------+-------+
|product_id|  product_name|       category|total_revenue|row_num|
+----------+--------------+---------------+-------------+-------+
|       105|  Cheese Block|          DAIRY|         8100|      1|
|       115|        Laptop|    ELECTRONICS|       186000|      1|
|       101|      Rice Bag|      GROCERIES|        42000|      1|
|       118|Water Purifier|HOME APPLIANCES|        48000|      1|
|       107|       Shampoo|  PERSONAL CARE|         8000|      1|
|       110|      Pen Pack|     STATIONERY|        13200|      1|
+----------+--------------+---------------+-------------+-------+



In [0]:
# 54
from pyspark.sql.functions import dense_rank
bill_window = Window.partitionBy("supplier_city").orderBy(col("bill_amount").desc())
final_df.withColumn("dense_rnk", dense_rank().over(bill_window)).select("supplier_name","supplier_city","bill_amount","dense_rnk").show()

+------------------+-------------+-----------+---------+
|     supplier_name|supplier_city|bill_amount|dense_rnk|
+------------------+-------------+-----------+---------+
|  National Grocers|    Ahmedabad|      18000|        1|
|        OfficeKart|    Bengaluru|      13200|        1|
|        OfficeKart|    Bengaluru|       6000|        2|
|     Kitchen World|      Chennai|      33000|        1|
|   Fresh Dairy Ltd|      Chennai|       8100|        2|
|   Fresh Dairy Ltd|      Chennai|       3000|        3|
| Elite Electronics|        Delhi|     186000|        1|
| Elite Electronics|        Delhi|     116000|        2|
| Elite Electronics|        Delhi|      90000|        3|
|     Reddy Traders|    Hyderabad|      31500|        1|
|     Reddy Traders|    Hyderabad|      24000|        2|
|  Daily Essentials|    Hyderabad|       8000|        3|
| Smart Electronics|      Kolkata|     125000|        1|
| Smart Electronics|      Kolkata|      84000|        2|
| Smart Electronics|      Kolka

In [0]:
# 55
supp_rev_df.withColumn("rank", rank().over(Window.orderBy(col("total_revenue").desc()))).filter(col("rank") <= 2).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-----------------+-------------+-------------+----+
|supplier_id|    supplier_name|supplier_city|total_revenue|rank|
+-----------+-----------------+-------------+-------------+----+
|        204|Elite Electronics|        Delhi|       392000|   1|
|        208|Smart Electronics|      Kolkata|       285000|   2|
+-----------+-----------------+-------------+-------------+----+



In [0]:
# 56
from pyspark.sql.functions import sum as spark_sum
date_window = Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
final_df.withColumn("running_total", spark_sum("bill_amount").over(date_window)).select("order_id","order_date","bill_amount","running_total").show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+----------+-----------+-------------+
|order_id|order_date|bill_amount|running_total|
+--------+----------+-----------+-------------+
|     301|2024-04-01|      24000|        24000|
|     302|2024-04-01|      31500|        55500|
|     303|2024-04-02|      90000|       145500|
|     304|2024-04-02|     125000|       270500|
|     305|2024-04-03|     186000|       456500|
|     306|2024-04-03|       3000|       459500|
|     307|2024-04-04|       8100|       467600|
|     308|2024-04-04|      24500|       492100|
|     309|2024-04-05|      48000|       540100|
|     310|2024-04-05|      33600|       573700|
|     311|2024-04-06|      33000|       606700|
|     312|2024-04-06|     116000|       722700|
|     313|2024-04-07|      84000|       806700|
|     314|2024-04-07|       6000|       812700|
|     315|2024-04-08|      13200|       825900|
|     316|2024-04-08|       7200|       833100|
|     317|2024-04-09|       8000|       841100|
|     318|2024-04-09|       3600|       

In [0]:
# 57
supp_window = Window.partitionBy("supplier_name").orderBy("order_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
final_df.withColumn("running_total", spark_sum("bill_amount").over(supp_window)).select("supplier_name","order_id","bill_amount","running_total").show()

+------------------+--------+-----------+-------------+
|     supplier_name|order_id|bill_amount|running_total|
+------------------+--------+-----------+-------------+
|CarePlus Suppliers|     316|       7200|         7200|
|CarePlus Suppliers|     318|       3600|        10800|
|  Daily Essentials|     317|       8000|         8000|
| Elite Electronics|     303|      90000|        90000|
| Elite Electronics|     305|     186000|       276000|
| Elite Electronics|     312|     116000|       392000|
|   Fresh Dairy Ltd|     306|       3000|         3000|
|   Fresh Dairy Ltd|     307|       8100|        11100|
| HomeNeeds Pvt Ltd|     308|      24500|        24500|
| HomeNeeds Pvt Ltd|     309|      48000|        72500|
| HomeNeeds Pvt Ltd|     310|      33600|       106100|
|     Kitchen World|     311|      33000|        33000|
|  National Grocers|     320|      18000|        18000|
|        OfficeKart|     314|       6000|         6000|
|        OfficeKart|     315|      13200|       

In [0]:
# 58
city_rev_df = final_df.groupBy("inventory_city").agg(sum("bill_amount").alias("total_revenue"))
city_rev_df.withColumn("rank", rank().over(Window.orderBy(col("total_revenue").desc()))).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------+-------------+----+
|inventory_city|total_revenue|rank|
+--------------+-------------+----+
|          Pune|       194000|   1|
|     Hyderabad|       173000|   2|
|     Bengaluru|       147500|   3|
|         Delhi|       146100|   4|
|       Chennai|       112000|   5|
|        Mumbai|        97200|   6|
|     Ahmedabad|        37200|   7|
|       Kolkata|        31700|   8|
+--------------+-------------+----+



In [0]:
# 59
cat_rev_df = final_df.groupBy("category").agg(sum("bill_amount").alias("total_revenue"))
cat_rev_df.withColumn("rank", rank().over(Window.orderBy(col("total_revenue").desc()))).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------------+-------------+----+
|       category|total_revenue|rank|
+---------------+-------------+----+
|    ELECTRONICS|       677000|   1|
|HOME APPLIANCES|       139100|   2|
|      GROCERIES|        73500|   3|
|     STATIONERY|        19200|   4|
|  PERSONAL CARE|        18800|   5|
|          DAIRY|        11100|   6|
+---------------+-------------+----+



In [0]:
# 60
mode_window = Window.partitionBy("payment_mode").orderBy(col("bill_amount").desc())
final_df.withColumn("row_num", row_number().over(mode_window)).filter(col("row_num") == 1).select("payment_mode","bill_amount","product_name","order_id").show()

+-------------+-----------+-------------+--------+
| payment_mode|bill_amount| product_name|order_id|
+-------------+-----------+-------------+--------+
|Bank Transfer|     186000|       Laptop|     305|
|         Cash|      33600|  Ceiling Fan|     310|
|  Credit Card|      33000|    Gas Stove|     311|
|   Debit Card|      24500|Mixer Grinder|     308|
|          UPI|     125000| Mobile Phone|     304|
+-------------+-----------+-------------+--------+



Part 7 — Delta Lake Core

In [0]:
# 61
final_df.write.format("delta").mode("overwrite").save(f"{DELTA_PATH}/retail_final")

In [0]:
# 62
from delta.tables import DeltaTable
from pyspark.sql.functions import to_date, lit

new_orders = spark.createDataFrame([
    (321, 114, 204, "2024-04-11", 3, "Delivered"),
    (322, 118, 210, "2024-04-11", 2, "Pending")
], ["order_id","product_id","supplier_id","order_date","quantity","order_status"])
new_orders = new_orders.withColumn("order_date", to_date("order_date","yyyy-MM-dd")) \
    .withColumn("order_priority", lit("High")) \
    .withColumn("low_stock_flag", lit("No"))
new_orders.write.format("delta").mode("append").save(f"{DELTA_PATH}/retail_final")

In [0]:
# 63
dt = DeltaTable.forPath(spark, f"{DELTA_PATH}/retail_final")
dt.update(
    condition="order_status = 'Pending' AND order_id = 304",
    set={"order_status": "'Delivered'"}
)

DataFrame[num_affected_rows: bigint]

In [0]:
# 64
dt.update(
    condition="payment_status = 'Pending' AND order_id = 304",
    set={"payment_status": "'Paid'"}
)

DataFrame[num_affected_rows: bigint]

In [0]:
# 65
dt.delete("order_status = 'Cancelled'")

DataFrame[num_affected_rows: bigint]

In [0]:
# 66
clean_df = spark.read.format("delta").load(f"{DELTA_PATH}/retail_final").filter(col("order_status") == "Delivered")
clean_df.write.format("delta").mode("overwrite").save(f"{DELTA_PATH}/clean_orders")

In [0]:
# 67
dt.history().show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------+---------------------------------------------------------------------------------+----+-----------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp          |userId         |userName                                               |operation|operationParameters                                                              |job |notebook         |queryHistoryStatementId             |clusterId               |readVersion|isolationLevel 

In [0]:
# 68
old_df = spark.read.format("delta").option("versionAsOf", 0).load(f"{DELTA_PATH}/retail_final")
old_df.show()

+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|     313|        208|       116|2024-04-07|  

In [0]:
# 69
latest_df = spark.read.format("delta").load(f"{DELTA_PATH}/retail_final")
print("Latest count:", latest_df.count())
print("Old version count:", old_df.count())
latest_df.show()
old_df.show()

Latest count: 20
Old version count: 20
+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|     3

In [0]:
# 70
spark.sql(f"VACUUM delta.`{DELTA_PATH}/retail_final` RETAIN 168 HOURS DRY RUN")

DataFrame[path: string]

Part 8 — Merge / Upsert

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import to_date

# 71
orders_df.write.format("delta").mode("overwrite").save(f"{DELTA_PATH}/target_orders")

# 72
target_dt = DeltaTable.forPath(spark, f"{DELTA_PATH}/target_orders")
target_dt.toDF().show()

# 73
daily_orders_data = [
(321,114,204,"2024-04-11",3,"Delivered"),
(322,118,210,"2024-04-11",2,"Delivered"),
(304,114,208,"2024-04-02",5,"Delivered"),
(319,112,208,"2024-04-10",2,"Delivered"),
(323,120,210,"2024-04-12",1,"Pending")
]
daily_orders_columns = ["order_id","product_id","supplier_id","order_date","quantity","order_status"]
daily_orders_df = spark.createDataFrame(daily_orders_data, daily_orders_columns)
daily_orders_df = daily_orders_df.withColumn("order_date", to_date("order_date","yyyy-MM-dd"))
daily_orders_df.createOrReplaceTempView("daily_orders")

# 74
target_dt.alias("target").merge(
    daily_orders_df.alias("source"),
    "target.order_id = source.order_id"
).whenMatchedUpdate(set={
    "order_status": "source.order_status",
    "quantity":     "source.quantity",
    "order_date":   "source.order_date"
}).whenNotMatchedInsert(values={
    "order_id":     "source.order_id",
    "product_id":   "source.product_id",
    "supplier_id":  "source.supplier_id",
    "order_date":   "source.order_date",
    "quantity":     "source.quantity",
    "order_status": "source.order_status"
}).execute()

print("Merge complete")
target_dt.toDF().show()

+----------+--------+-----------+----------+--------+------------+--------------+
|product_id|order_id|supplier_id|order_date|quantity|order_status|order_priority|
+----------+--------+-----------+----------+--------+------------+--------------+
|       116|     313|        208|2024-04-07|       2|     Pending|          High|
|       109|     314|        205|2024-04-07|      80|   Delivered|           Low|
|       110|     315|        205|2024-04-08|     120|   Delivered|           Low|
|       111|     303|        204|2024-04-02|       2|   Delivered|           Low|
|       114|     304|        208|2024-04-02|       5|     Pending|          High|
|       115|     305|        204|2024-04-03|       3|   Delivered|           Low|
|       117|     308|        206|2024-04-04|       7|   Delivered|           Low|
|       118|     309|        206|2024-04-05|       4|     Pending|          High|
|       119|     310|        206|2024-04-05|      12|   Delivered|           Low|
|       108|    

In [0]:
# 75
target_dt.toDF().filter(col("order_id").isin(304, 319)).show()

+----------+--------+-----------+----------+--------+------------+--------------+
|product_id|order_id|supplier_id|order_date|quantity|order_status|order_priority|
+----------+--------+-----------+----------+--------+------------+--------------+
|       114|     304|        208|2024-04-02|       5|   Delivered|          High|
|       112|     319|        208|2024-04-10|       2|   Delivered|          High|
+----------+--------+-----------+----------+--------+------------+--------------+



In [0]:
# 76
target_dt.toDF().filter(col("order_id").isin(321, 322, 323)).show()

+----------+--------+-----------+----------+--------+------------+--------------+
|product_id|order_id|supplier_id|order_date|quantity|order_status|order_priority|
+----------+--------+-----------+----------+--------+------------+--------------+
|       114|     321|        204|2024-04-11|       3|   Delivered|          NULL|
|       118|     322|        210|2024-04-11|       2|   Delivered|          NULL|
|       120|     323|        210|2024-04-12|       1|     Pending|          NULL|
+----------+--------+-----------+----------+--------+------------+--------------+



In [0]:
# 77
target_dt.toDF().filter(col("order_id").isin(304, 319)).show()

+----------+--------+-----------+----------+--------+------------+--------------+
|product_id|order_id|supplier_id|order_date|quantity|order_status|order_priority|
+----------+--------+-----------+----------+--------+------------+--------------+
|       114|     304|        208|2024-04-02|       5|   Delivered|          High|
|       112|     319|        208|2024-04-10|       2|   Delivered|          High|
+----------+--------+-----------+----------+--------+------------+--------------+



In [0]:
# 78
target_dt.toDF().filter(col("order_id").isin(321, 322, 323)).show()

+----------+--------+-----------+----------+--------+------------+--------------+
|product_id|order_id|supplier_id|order_date|quantity|order_status|order_priority|
+----------+--------+-----------+----------+--------+------------+--------------+
|       114|     321|        204|2024-04-11|       3|   Delivered|          NULL|
|       118|     322|        210|2024-04-11|       2|   Delivered|          NULL|
|       120|     323|        210|2024-04-12|       1|     Pending|          NULL|
+----------+--------+-----------+----------+--------+------------+--------------+



In [0]:
# 79
target_dt.history().show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+-----------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
# 80
print("""
MERGE handles both INSERT and UPDATE in a single atomic operation.
It prevents duplicates, avoids full rewrites, and maintains Delta history.
Essential for incremental loads in real-world pipelines.
""")


MERGE handles both INSERT and UPDATE in a single atomic operation.
It prevents duplicates, avoids full rewrites, and maintains Delta history.
Essential for incremental loads in real-world pipelines.



Part 9 — Parquet to Delta

In [0]:
# 81
products_df.write.mode("overwrite").parquet(f"{PARQUET_PATH}/products")

In [0]:
# 82
parquet_df = spark.read.parquet(f"{PARQUET_PATH}/products")
parquet_df.show()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+
|product_id|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+
|       118| Water Purifier|HOME APPLIANCES|         Delhi|12000|            20|      Medium|                   Yes|         240000|            No|
|       119|    Ceiling Fan|HOME APPLIANCES|     Ahmedabad| 2800|            60|        High|                    No|         168000|            No|
|       120|      Gas Stove|HOME APPLIANCES|       Chennai| 5500|            25|      Medium|                    No|         137500|            No|
|       113|Washing Machine|    ELECTRONICS|     Bengaluru|29000|            12|         Low|                   

In [0]:
# 83
parquet_df.write.format("delta").mode("overwrite").save(f"{DELTA_PATH}/products_delta")

In [0]:
# 84
delta_products = spark.read.format("delta").load(f"{DELTA_PATH}/products_delta")
delta_products.show()
delta_products.printSchema()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+
|product_id|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|        High|                    No|          60000|            No|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|        High|                    No|          72000|            No|
|       103|  Sunflower Oil|      GROCERIES|        Mumbai| 1800|            40|      Medium|                    No|          72000|            No|
|       104|      Milk Pack|          DAIRY|       Chennai|   60|           200|        High|                   

In [0]:
# 85
print("Parquet: Immutable, no updates, no history, no ACID")
print("Delta: ACID, update/delete, time travel, schema enforcement")

Parquet: Immutable, no updates, no history, no ACID
Delta: ACID, update/delete, time travel, schema enforcement


In [0]:
# 86
dt_products = DeltaTable.forPath(spark, f"{DELTA_PATH}/products_delta")
dt_products.update(
    condition="category = 'ELECTRONICS'",
    set={"stock_status": "'Low'"}
)
dt_products.toDF().show()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+
|product_id|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|low_stock_flag|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+--------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|        High|                    No|          60000|            No|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|        High|                    No|          72000|            No|
|       103|  Sunflower Oil|      GROCERIES|        Mumbai| 1800|            40|      Medium|                    No|          72000|            No|
|       104|      Milk Pack|          DAIRY|       Chennai|   60|           200|        High|                   

In [0]:
# 87
print("""
Delta uses transaction logs (_delta_log) to track all changes.
Updates rewrite only affected files, not the entire dataset.
Parquet requires full file rewrites for any update — no partial edits possible.
""")


Delta uses transaction logs (_delta_log) to track all changes.
Updates rewrite only affected files, not the entire dataset.
Parquet requires full file rewrites for any update — no partial edits possible.



In [0]:
# cleanup — run once before running DLT Retail pipeline
spark.sql("DROP TABLE IF EXISTS hexa_ws_7405609620479942.default.gold_city_revenue")
spark.sql("DROP TABLE IF EXISTS hexa_ws_7405609620479942.default.gold_category_revenue")
spark.sql("DROP TABLE IF EXISTS hexa_ws_7405609620479942.default.silver_orders")
spark.sql("DROP TABLE IF EXISTS hexa_ws_7405609620479942.default.bronze_orders")

DataFrame[]

In [0]:
spark.sql("SHOW SCHEMAS IN hexa_ws_7405609620479942").show()

+------------------+
|      databaseName|
+------------------+
|           default|
| delta_training_db|
|information_schema|
|  training_sql_lab|
+------------------+



In [0]:
# 95
spark.sql("SHOW TABLES IN hexa_ws_7405609620479942.default").show()
spark.sql("SELECT * FROM hexa_ws_7405609620479942.default.bronze_orders").show()
spark.sql("SELECT * FROM hexa_ws_7405609620479942.default.silver_orders").show()
spark.sql("SELECT * FROM hexa_ws_7405609620479942.default.gold_city_revenue").show()
spark.sql("SELECT * FROM hexa_ws_7405609620479942.default.gold_category_revenue").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|     bronze_hospital|      false|
| default|       bronze_orders|      false|
| default|bronze_patient_vi...|      false|
| default| bronze_sales_inline|      false|
| default|    ecommerce_orders|      false|
| default|gold_category_rev...|      false|
| default|   gold_city_revenue|      false|
| default|gold_city_sales_s...|      false|
| default|gold_department_s...|      false|
| default|gold_hospital_sum...|      false|
| default|gold_specializati...|      false|
| default|     hospital_visits|      false|
| default|hospital_visits_d...|      false|
| default|hospital_visits_p...|      false|
| default|hospital_visits_t...|      false|
| default|     silver_hospital|      false|
| default|       silver_orders|      false|
| default|silver_patient_vi...|      false|
| default|silver_sales_cleaned|      false|
|        |        daily_orders| 

Part 11 — Unity Catalog and Governance

In [0]:
# 96
spark.sql("SHOW CATALOGS").show()
print("Using existing catalog: hexa_ws_7405609620479942")
print("Catalog already exists — no CREATE CATALOG permission needed in shared workspace")

+--------------------+
|             catalog|
+--------------------+
|          assessment|
|              bronze|
|               datas|
|hexa_ws_740560962...|
| hexacatalog_student|
|    hospital_catalog|
|     hospitalcatalog|
|                main|
|           medallion|
|             samples|
|      sensor_catalog|
|              system|
|              task_1|
+--------------------+

Using existing catalog: hexa_ws_7405609620479942
Catalog already exists — no CREATE CATALOG permission needed in shared workspace


In [0]:
# 97
spark.sql("CREATE SCHEMA IF NOT EXISTS hexa_ws_7405609620479942.supply_chain")
spark.sql("DESCRIBE SCHEMA hexa_ws_7405609620479942.supply_chain").show()

+-------------------------+--------------------------+
|database_description_item|database_description_value|
+-------------------------+--------------------------+
|             Catalog Name|      hexa_ws_740560962...|
|           Namespace Name|              supply_chain|
|                  Comment|                          |
|                 Location|                          |
|                    Owner|      azuser5814_mml.lo...|
+-------------------------+--------------------------+



In [0]:
# 98
final_df.write.format("delta").mode("overwrite") \
    .saveAsTable("hexa_ws_7405609620479942.supply_chain.retail_final")
spark.sql("DESCRIBE TABLE hexa_ws_7405609620479942.supply_chain.retail_final").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|            order_id|   bigint|   NULL|
|         supplier_id|   bigint|   NULL|
|          product_id|   bigint|   NULL|
|          order_date|     date|   NULL|
|            quantity|   bigint|   NULL|
|        order_status|   string|   NULL|
|      order_priority|   string|   NULL|
|        product_name|   string|   NULL|
|            category|   string|   NULL|
|      inventory_city|   string|   NULL|
|               price|   bigint|   NULL|
|      stock_quantity|   bigint|   NULL|
|        stock_status|   string|   NULL|
|expensive_product...|   string|   NULL|
|     inventory_value|   bigint|   NULL|
|      low_stock_flag|   string|   NULL|
|       supplier_name|   string|   NULL|
|       supplier_city|   string|   NULL|
|      specialization|   string|   NULL|
|          payment_id|   bigint|   NULL|
+--------------------+---------+-------+
only showing top

In [0]:
# 99
spark.sql("""
CREATE OR REPLACE TABLE hexa_ws_7405609620479942.supply_chain.revenue_by_category AS
SELECT category, SUM(bill_amount) AS total_revenue
FROM hexa_ws_7405609620479942.supply_chain.retail_final
GROUP BY category
""")
spark.sql("SELECT * FROM hexa_ws_7405609620479942.supply_chain.revenue_by_category").show()
print("Lineage: retail_final → revenue_by_category (visible in Catalog > Lineage tab)")

+---------------+-------------+
|       category|total_revenue|
+---------------+-------------+
|     STATIONERY|        19200|
|    ELECTRONICS|       677000|
|HOME APPLIANCES|       139100|
|      GROCERIES|        73500|
|  PERSONAL CARE|        18800|
|          DAIRY|        11100|
+---------------+-------------+

Lineage: retail_final → revenue_by_category (visible in Catalog > Lineage tab)


In [0]:
# 100
spark.sql("""
GRANT SELECT ON TABLE hexa_ws_7405609620479942.supply_chain.retail_final
TO `azuser5814_mml.local@karthikirisoutlook.onmicrosoft.com`
""")
print("""
Governance behavior:
- GRANT SELECT allows user to only READ the table
- Cannot INSERT, UPDATE or DELETE
- Unity Catalog enforces at metastore level
- All access is logged and auditable via system tables
- Row and column level security can be added on top
""")


Governance behavior:
- GRANT SELECT allows user to only READ the table
- Cannot INSERT, UPDATE or DELETE
- Unity Catalog enforces at metastore level
- All access is logged and auditable via system tables
- Row and column level security can be added on top

